# 02 — Drift Detection Demo

Demonstrates statistical drift detection on real Census ACS data.

**Tests shown:**
- Population Stability Index (PSI)
- Kolmogorov-Smirnov (KS) test
- Alert classification (stable / warning / critical)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.model_registry import ModelRegistry
from src.drift_detector import DriftDetector, calculate_psi, ks_test
from src.retrain_pipeline import fetch_census_acs
import pandas as pd
import numpy as np

# Setup
reg = ModelRegistry('../data/registry.db')
detector = DriftDetector(reg)
print("Drift detector ready")

## Fetch Real Census Data

In [ ]:
# Fetch fresh real data from Census API
df = fetch_census_acs()
print(f"Fetched {len(df)} records")
print(df[['state_name', 'median_income', 'education_rate', 'poverty_rate']].head())

## Run Drift Detection

In [ ]:
# Run drift detection against production baseline
features = ['population', 'median_age', 'education_rate', 'poverty_rate']
result = detector.detect_drift_for_experiment('census-demographics', df, features)

print(f"Overall alert: {result['overall_alert'].upper()}")
print("\nFeature-level results:")
for feat, res in result['features'].items():
    print(f"  {feat}:")
    print(f"    PSI: {res['psi']:.4f} ({res['psi_status']})")
    print(f"    KS p-value: {res['ks_p_value']:.4f} ({res['ks_status']})")
    print(f"    Expected μ={res['expected_mean']:.4f}, Actual μ={res['actual_mean']:.4f}")

## Manual PSI/KS Test on Single Feature

In [ ]:
# Manual test: compare two subsets of the data
subset_a = df[df['median_income'] > df['median_income'].median()]['education_rate'].dropna().values
subset_b = df[df['median_income'] <= df['median_income'].median()]['education_rate'].dropna().values

psi = calculate_psi(subset_a, subset_b)
ks_stat, ks_p = ks_test(subset_a, subset_b)

print(f"Education rate PSI between high/low income states: {psi:.4f}")
print(f"KS test: statistic={ks_stat:.4f}, p-value={ks_p:.4f}")